# LIDS Demo

This notebook demonstrates the similarity metric portion of **LIDS (LLM Summary Inference Under the Layered Lens)** as presented in the [LIDS paper](https://arxiv.org/abs/2603.00105):

1. Convert texts into BERT token embedding matrices.
2. Compute the singular value decomposition (SVD) of the token embedding matrices.
3. Construct the LIDS direction vectors across the leading SVD layers.
4. Compare texts using the maximum absolute cosine similarity over aligned layers.
5. Return the maximum absolute cosine similarity as the final LIDS similarity score.

The examples are self contained and require no research dataset.

## 1. Install dependencies

From the repository root, install the dependencies with:

```bash
python -m pip install -r requirements.txt
```

The notebook assumes the packages in `requirements.txt` are already installed.

In [1]:
import numpy as np
import torch

from transformers import BertModel, BertTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch version: 2.9.1
Using device: cpu


## 2. Load BERT

LIDS begins with token-level BERT representations. `bert-base-uncased` produces a 768-dimensional embedding for each retained token, so each text becomes an embedding matrix $X_j$ with one row per token.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.to(device)
model.eval()

print("BERT loaded successfully.")
print(f"Hidden dimension: {model.config.hidden_size}")

BERT loaded successfully.
Hidden dimension: 768


## 3. Define the example texts

For a simple example we will have four texts. One as a reference and three to compare to it.

The first text wil be our reference text that we will compare all other texts to. This text is about gardening and is four sentences long. The second text is a one sentence summary that uses similar terminology and phrasing as the reference text. The third text is a one sentence summary using different vocabulary and phrasing, but contains the same information. The fourth text is an unrelated sentence about spacecraft. 

In [3]:
texts = [
    # 1. Reference text: longer reference
    (
        "Successful gardening requires regular attention to several important factors. "
        "Plants need healthy soil, sufficient sunlight, and consistent watering to grow properly throughout the season. "
        "Gardeners must also monitor their plants, remove weeds, and prune when necessary to prevent problems and maintain healthy growth. "
        "Adjusting care based on seasonal conditions can help keep a garden productive and healthy."
    ),

    # 2. Summary: similar terminology and phrasing
    "Successful gardening requires healthy soil, sufficient sunlight, consistent watering, and regular maintenance such as removing weeds and pruning.",

    # 3. Summary: different vocabulary, same information
    "A thriving garden depends on fertile ground, adequate light, reliable hydration, and routine upkeep to support plant growth throughout the year.",

    # 4. Completely unrelated
    "Researchers developed a new spacecraft navigation system that uses onboard sensors to calculate trajectories and make precise adjustments during long distance missions.",
]

text_names = [
    "reference",
    "garden summary similar vocabulary",
    "garden summary differing vocabulary",
    "unrelated to reference",
]

for name, text in zip(text_names, texts):
    print(f"{name}: {text}\n")

reference: Successful gardening requires regular attention to several important factors. Plants need healthy soil, sufficient sunlight, and consistent watering to grow properly throughout the season. Gardeners must also monitor their plants, remove weeds, and prune when necessary to prevent problems and maintain healthy growth. Adjusting care based on seasonal conditions can help keep a garden productive and healthy.

garden summary similar vocabulary: Successful gardening requires healthy soil, sufficient sunlight, consistent watering, and regular maintenance such as removing weeds and pruning.

garden summary differing vocabulary: A thriving garden depends on fertile ground, adequate light, reliable hydration, and routine upkeep to support plant growth throughout the year.

unrelated to reference: Researchers developed a new spacecraft navigation system that uses onboard sensors to calculate trajectories and make precise adjustments during long distance missions.



## 4. Construct the LIDS direction vectors

For text $T_j$ with BERT embedding matrix $X_j$, compute the SVD

$$X_j = U_j \Sigma_j V_j^T.$$

Following equation (3) in the paper, define the sign correction

$$s_{jl} = \operatorname{sign}(\langle v_{jl}, p^{-1/2}\mathbf{1}_p\rangle),$$

where $p$ is the embedding dimension. The cumulative direction vector

$$d_j(k) = \sum_{l=1}^{k} \lambda_{jl}^{\alpha}s_{jl}X_j^T u_{jl}.$$

The code below stores $d_j(1), \ldots, d_j(k_{\max})$. We use $\alpha=1$ and at most 11 layers for this compact demonstration.

In [4]:
def compute_direction_vectors(
    text,
    tokenizer,
    model,
    k_layers=11,
    alpha=1.0,
    max_length=512,
    device=None,
):
    """Compute cumulative LIDS direction vectors for one text."""
    if device is None:
        device = next(model.parameters()).device

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        add_special_tokens=True,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    embeddings = outputs.last_hidden_state[0]
    attention_mask = encoded["attention_mask"][0].bool()
    input_ids = encoded["input_ids"][0]
    special_mask = tokenizer.get_special_tokens_mask(
        input_ids.tolist(), already_has_special_tokens=True
    )
    special_mask = torch.tensor(special_mask, device=device, dtype=torch.bool)

    keep = attention_mask & ~special_mask
    X = embeddings[keep].detach().cpu().numpy().astype(np.float64)
    if X.shape[0] == 0:
        raise ValueError("The text produced no non-special BERT tokens.")

    U, singular_values, Vt = np.linalg.svd(X, full_matrices=False)
    V = Vt.T
    _, p = X.shape
    k_max = min(k_layers, *X.shape)

    direction_vectors = []
    direction = np.zeros(p, dtype=np.float64)
    ones_normalized = np.ones(p, dtype=np.float64) / np.sqrt(p)

    for layer in range(k_max):
        sign = np.sign(np.dot(V[:, layer], ones_normalized))
        if sign == 0:
            sign = 1.0

        direction += (
            singular_values[layer] ** alpha
            * sign
            * (X.T @ U[:, layer])
        )
        direction_vectors.append(direction.copy())

    return direction_vectors, X, singular_values[:k_max]

## 5. Compute each text's direction vectors

Algorithm 1 first constructs the BERT-SVD direction vectors separately for the reference and every test text. 

In [5]:
k_layers = 11
alpha = 1.0
max_length = 512

direction_vector_sets = []
embedding_matrices = []
singular_value_sets = []

for name, text in zip(text_names, texts):
    direction_vectors, embedding_matrix, singular_values = compute_direction_vectors(
        text=text,
        tokenizer=tokenizer,
        model=model,
        k_layers=k_layers,
        alpha=alpha,
        max_length=max_length,
        device=device,
    )
    direction_vector_sets.append(direction_vectors)
    embedding_matrices.append(embedding_matrix)
    singular_value_sets.append(singular_values)
    print(
        f"{name}: matrix {embedding_matrix.shape}, "
        f"{len(direction_vectors)} retained layers"
    )

reference: matrix (67, 768), 11 retained layers
garden summary similar vocabulary: matrix (24, 768), 11 retained layers
garden summary differing vocabulary: matrix (28, 768), 11 retained layers
unrelated to reference: matrix (26, 768), 11 retained layers


## 6. LIDS similarity

For test text $T_j$ and reference $T_0$, equation (4) defines the LIDS similarity metric as

$$\operatorname{MACS}_j = \max_k \left|\operatorname{CS}(d_j(k),d_0(k))\right|.$$

The layer count is aligned between the two texts. The maximizing layer $\hat{k}_j$ identifies both the reported similarity and the summary embedding $d_j(\hat{k}_j)$.

In [6]:
def cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        raise ValueError("Cannot compare a zero direction vector.")
    return float(np.dot(vec1, vec2) / (norm1 * norm2))


def compare_with_reference(reference_directions, test_directions):
    """Return MACS, optimal layer, layer scores, and the test embedding."""
    num_layers = min(len(reference_directions), len(test_directions))
    if num_layers == 0:
        raise ValueError("Both texts must have at least one direction vector.")

    layer_similarities = [
        abs(cosine_similarity(reference_directions[k], test_directions[k]))
        for k in range(num_layers)
    ]
    best_index = int(np.argmax(layer_similarities))

    return {
        "similarity": layer_similarities[best_index],
        "best_k": best_index + 1,
        "layer_similarities": layer_similarities,
        "summary_embedding": test_directions[best_index].copy(),
    }

## 7. Calculate the LIDS similarity scores

We now compare the similarity between the reference text and the test texts using the LIDS similarity metric we just defined.

In [7]:
reference_directions = direction_vector_sets[0]
results = {}

for test_index in range(1, len(texts)):
    name = text_names[test_index]
    result = compare_with_reference(
        reference_directions,
        direction_vector_sets[test_index],
    )
    results[name] = result
    print(f"{name}")
    print(f"  LIDS similarity (MACS): {result['similarity']:.6f}")
    print(f"  optimal layer (k-hat):   {result['best_k']}")
    print(f"  summary embedding shape: {result['summary_embedding'].shape} \n")

garden summary similar vocabulary
  LIDS similarity (MACS): 0.918987
  optimal layer (k-hat):   1
  summary embedding shape: (768,) 

garden summary differing vocabulary
  LIDS similarity (MACS): 0.912336
  optimal layer (k-hat):   1
  summary embedding shape: (768,) 

unrelated to reference
  LIDS similarity (MACS): 0.664781
  optimal layer (k-hat):   1
  summary embedding shape: (768,) 



## 8. Interpret the three comparisons

- **Similar vocabulary:** this summary should score highly because it preserves the reference's subject, facts, and much of its wording.
- **Different vocabulary:** this summary should also score highly if LIDS captures semantic agreement beyond exact word overlap.
- **Unrelated text:** the spacecraft sentence should score lower because it does not summarize the gardening reference.

These are illustrative single comparisons. The paper studies statistical uncertainty using repeated LLM-generated summaries vs. benchmarks. The paper also studies LIDS correlation to human evaluations of summary qualities. Such analysis is beyond this compact demo.

In [8]:
scores = np.array([result["similarity"] for result in results.values()])
assert np.all((0.0 <= scores) & (scores <= 1.0))
assert all(result["summary_embedding"].shape == (768,) for result in results.values())

print("All scores are in [0, 1], and all LIDS summary embeddings are 768-dimensional.")

All scores are in [0, 1], and all LIDS summary embeddings are 768-dimensional.


## 9. Use the reusable package

The cells above expose the algorithm. In normal use, the repository's public function performs the same pipeline with two required text inputs:

```python
from lids import LIDS_similarity

score = LIDS_similarity(text1, text2)
```

See `LIDS_quickstart.ipynb` for the shortest path from installation to your first comparison.

## What this demonstrates

**Reference and test texts -> BERT embeddings -> SVD layers -> cumulative direction vectors -> maximum absolute cosine similarity (the LIDS similarity score).**

This mirrors the reference to summary workflow in equations (3)-(4) and Algorithm 1 of the [LIDS paper on arXiv](https://arxiv.org/abs/2603.00105).